In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from scipy import stats
import time
import os

ea_dir = r"C:\Users\user\Downloads\GSE148812_clean"

# Reload gene burden matrix
gene_burden_cpd_ea_pc = pd.read_csv(os.path.join(ea_dir, "gene_burden_matrix_signed_protein_coding_CPD_EA.csv"), index_col=0)
gene_names_cpd_ea = gene_burden_cpd_ea_pc.index.tolist()
smoker_cols_ea_cpd = gene_burden_cpd_ea_pc.columns.tolist()

# Reload phenotype
meta_df_ea = pd.read_csv(os.path.join(ea_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))
meta_df_ea["sample_id"] = meta_df_ea["sample_id"].astype(str)
meta_df_ea = meta_df_ea.set_index("sample_id")
pheno_cpd_ea = meta_df_ea.loc[smoker_cols_ea_cpd, "cpd"].astype(float)
print("EA CPD phenotype reloaded:", pheno_cpd_ea.shape)

# Reload rebuilt confounders (the trustworthy one we built ourselves)
confounders_cpd_ea_rebuilt = np.load(os.path.join(ea_dir, "confounders_X_cpd_EA_rebuilt.npy"))
print("Rebuilt confounders reloaded:", confounders_cpd_ea_rebuilt.shape)

# DoubleML setup
X_genes_cpd_ea = gene_burden_cpd_ea_pc.T.values
X_standardized_cpd_ea = (X_genes_cpd_ea - X_genes_cpd_ea.mean(axis=0)) / X_genes_cpd_ea.std(axis=0)
Y_cpd_ea = pheno_cpd_ea.values

def doubleml_scan(X_snps, Y, X_conf, n_folds=5, random_state=42):
    n, n_snps = X_snps.shape
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    D_resid = np.zeros_like(X_snps)
    Y_resid = np.zeros(n)
    Xc = np.column_stack([np.ones(n), X_conf])

    for train_idx, test_idx in kf.split(Xc):
        Xc_tr, Xc_te = Xc[train_idx], Xc[test_idx]
        coef_Y = np.linalg.lstsq(Xc_tr, Y[train_idx], rcond=None)[0]
        Y_resid[test_idx] = Y[test_idx] - Xc_te @ coef_Y
        coef_D = np.linalg.lstsq(Xc_tr, X_snps[train_idx], rcond=None)[0]
        D_resid[test_idx] = X_snps[test_idx] - Xc_te @ coef_D

    Yr = Y_resid - Y_resid.mean()
    Dr = D_resid - D_resid.mean(axis=0)
    del D_resid

    sum_DY = (Dr * Yr[:, None]).sum(axis=0)
    sum_DD = (Dr ** 2).sum(axis=0)
    sum_YY = (Yr ** 2).sum()

    theta = sum_DY / sum_DD
    ssr = sum_YY - (sum_DY ** 2) / sum_DD
    sigma2 = ssr / (n - 2)
    se = np.sqrt(sigma2 / sum_DD)
    t_stat = theta / se
    pvals = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 2))
    return theta, pvals

n_repeats = 30
threshold = 0.001
n_genes_cpd_ea = X_standardized_cpd_ea.shape[1]
significant_counts_cpd_ea = np.zeros(n_genes_cpd_ea, dtype=int)

start = time.time()
for rep in range(n_repeats):
    theta_rep, pval_rep = doubleml_scan(X_standardized_cpd_ea, Y_cpd_ea, confounders_cpd_ea_rebuilt, random_state=rep)
    significant_counts_cpd_ea += (pval_rep < threshold).astype(int)
    if (rep + 1) % 5 == 0:
        print(f"Completed {rep+1}/{n_repeats}, elapsed {time.time()-start:.1f}s")

print(f"Total time: {time.time()-start:.1f}s")

stability_fraction_cpd_ea = significant_counts_cpd_ea / n_repeats
results_df_cpd_ea = pd.DataFrame({
    "gene": gene_names_cpd_ea,
    "stability_fraction": stability_fraction_cpd_ea,
    "n_significant_repeats": significant_counts_cpd_ea
}).sort_values("stability_fraction", ascending=False)

print("\nStability distribution:")
for t in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    print(f"  >= {t:.0%}: {(stability_fraction_cpd_ea >= t).sum()} genes")

results_df_cpd_ea.to_csv(os.path.join(ea_dir, "gene_doubleml_stability_cpd_EA.csv"), index=False)
print("\nSaved.")

EA CPD phenotype reloaded: (793,)
Rebuilt confounders reloaded: (793, 12)
Completed 5/30, elapsed 3.1s
Completed 10/30, elapsed 6.1s
Completed 15/30, elapsed 9.1s
Completed 20/30, elapsed 12.2s
Completed 25/30, elapsed 15.2s
Completed 30/30, elapsed 18.2s
Total time: 18.2s

Stability distribution:
  >= 50%: 406 genes
  >= 60%: 271 genes
  >= 70%: 171 genes
  >= 80%: 88 genes
  >= 90%: 42 genes
  >= 100%: 10 genes

Saved.


In [3]:
import time
import json
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode

shortlist_100_cpd_ea = results_df_cpd_ea[results_df_cpd_ea["stability_fraction"] == 1.0].copy()
print("EA CPD genes at 100%:", shortlist_100_cpd_ea["gene"].tolist())

gene_burden_shortlist_cpd_ea = gene_burden_cpd_ea_pc.loc[shortlist_100_cpd_ea["gene"]]
corr_cpd_ea = gene_burden_shortlist_cpd_ea.T.corr()
r2_cpd_ea = (corr_cpd_ea ** 2).values.copy()
np.fill_diagonal(r2_cpd_ea, 0)
dup_pairs_ea = np.argwhere(r2_cpd_ea > 0.99)

dup_drop_ea = set()
for i, j in dup_pairs_ea:
    if i < j:
        g1, g2 = shortlist_100_cpd_ea["gene"].iloc[i], shortlist_100_cpd_ea["gene"].iloc[j]
        print(f"Duplicate: {g1} <-> {g2}, r²={r2_cpd_ea[i,j]:.4f}")
        dup_drop_ea.add(g2)

shortlist_dedup_cpd_ea = shortlist_100_cpd_ea[~shortlist_100_cpd_ea["gene"].isin(dup_drop_ea)]
print(f"\nEA CPD genes after dedup: {len(shortlist_dedup_cpd_ea)}")

genes_list_cpd_ea = shortlist_dedup_cpd_ea.sort_values("stability_fraction", ascending=False)["gene"].tolist()
gene_burden_final_cpd_ea = gene_burden_cpd_ea_pc.loc[genes_list_cpd_ea]
X_genes_pc_cpd_ea = gene_burden_final_cpd_ea.T.values
Y_pc_cpd_ea = pheno_cpd_ea.values.reshape(-1, 1)
X_pc_full_cpd_ea = np.hstack([X_genes_pc_cpd_ea, Y_pc_cpd_ea])
col_names_cpd_ea = gene_burden_final_cpd_ea.index.tolist() + ["cpd"]

print("EA CPD PC input shape:", X_pc_full_cpd_ea.shape)

n_nodes_cpd_ea = len(col_names_cpd_ea)
outcome_idx_cpd_ea = col_names_cpd_ea.index("cpd")

bk_cpd_ea = BackgroundKnowledge()
nodes_cpd_ea = [GraphNode(name) for name in col_names_cpd_ea]
for i in range(n_nodes_cpd_ea - 1):
    bk_cpd_ea.add_node_to_tier(nodes_cpd_ea[i], 0)
bk_cpd_ea.add_node_to_tier(nodes_cpd_ea[outcome_idx_cpd_ea], 1)

start = time.time()
cg_cpd_ea = pc(
    data=X_pc_full_cpd_ea,
    alpha=0.001,
    indep_test=fisherz,
    stable=True,
    uc_rule=0,
    uc_priority=2,
    background_knowledge=bk_cpd_ea,
    depth=3,
    verbose=False,
    show_progress=True,
    node_names=col_names_cpd_ea
)
elapsed = time.time() - start
print(f"Completed in {elapsed:.1f}s")

adj_cpd_ea = cg_cpd_ea.G.graph
direct_parents_cpd_ea = [col_names_cpd_ea[i] for i in range(n_nodes_cpd_ea) if col_names_cpd_ea[i] != "cpd"
                          and ((adj_cpd_ea[i, outcome_idx_cpd_ea] == -1 and adj_cpd_ea[outcome_idx_cpd_ea, i] == 1)
                               or (adj_cpd_ea[i, outcome_idx_cpd_ea] == -1 and adj_cpd_ea[outcome_idx_cpd_ea, i] == -1))]
print(f"\nEA CPD direct parents: {len(direct_parents_cpd_ea)}")
print(direct_parents_cpd_ea)

# Check against AA CPD's direct parents
aa_cpd_parents = ['NCKAP5', 'EXD3', 'TRIM66', 'HYDIN', 'COL18A1', 'ZNF805', 'THNSL2', 'PPP1R12B', 'FAT4']
overlap_cpd = set(direct_parents_cpd_ea) & set(aa_cpd_parents)
print(f"\nOverlap with AA CPD direct parents: {overlap_cpd}")

c:\Users\user\Desktop\ai causal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


EA CPD genes at 100%: ['SASH1', 'PLXNA2', 'ESYT3', 'LYAR', 'LAMA1', 'SVEP1', 'SYCE1L', 'NIPA1', 'CPM', 'AHRR']

EA CPD genes after dedup: 10
EA CPD PC input shape: (793, 11)


Depth=2, working on node 10: 100%|██████████| 11/11 [00:00<00:00, 1034.05it/s]

Completed in 0.0s

EA CPD direct parents: 3
['PLXNA2', 'LAMA1', 'SVEP1']

Overlap with AA CPD direct parents: set()


In [4]:
aa_cpd_100 = set(pd.read_csv(r"C:\Users\user\Downloads\GSE148375_clean\gene_doubleml_stability_cpd_AA.csv")
                  .pipe(lambda d: d[d["stability_fraction"]==1.0])["gene"].tolist())
ea_cpd_100 = set(shortlist_100_cpd_ea["gene"].tolist())

print("AA CPD genes at 100%:", len(aa_cpd_100))
print("EA CPD genes at 100%:", len(ea_cpd_100))

overlap_cpd_pre_pc = aa_cpd_100 & ea_cpd_100
print(f"\nOverlap BEFORE PC algorithm: {len(overlap_cpd_pre_pc)}")
print(sorted(overlap_cpd_pre_pc))

from scipy.stats import hypergeom
M = 11663  # AA CPD protein-coding genes tested (approx population size; EA tested fewer but AA is close to full)
n = len(aa_cpd_100)
N = len(ea_cpd_100)
k = len(overlap_cpd_pre_pc)
if k > 0:
    p = hypergeom.sf(k-1, M, n, N)
    print(f"\nP(>= {k} overlap by chance): {p:.6f}")
else:
    print("\nNo overlap even at DoubleML stage.")

AA CPD genes at 100%: 33
EA CPD genes at 100%: 10

Overlap BEFORE PC algorithm: 0
[]

No overlap even at DoubleML stage.


In [5]:
import pandas as pd

aa_cpd_all = pd.read_csv(r"C:\Users\user\Downloads\GSE148375_clean\gene_doubleml_stability_cpd_AA.csv")
ea_cpd_all = results_df_cpd_ea  # already in memory from EA run

aa_cpd_90 = set(aa_cpd_all[aa_cpd_all["stability_fraction"] >= 0.9]["gene"].tolist())
ea_cpd_90 = set(ea_cpd_all[ea_cpd_all["stability_fraction"] >= 0.9]["gene"].tolist())

print("AA CPD genes at >=90%:", len(aa_cpd_90))
print("EA CPD genes at >=90%:", len(ea_cpd_90))

overlap_90 = aa_cpd_90 & ea_cpd_90
print(f"\nOverlap at >=90% stability: {len(overlap_90)}")
print(sorted(overlap_90))

from scipy.stats import hypergeom
M = 11663  # approx AA protein-coding genes tested
n = len(aa_cpd_90)
N = len(ea_cpd_90)
k = len(overlap_90)
if k > 0:
    p = hypergeom.sf(k-1, M, n, N)
    print(f"\nP(>= {k} overlap by chance): {p:.6f}")
else:
    print("\nStill no overlap at 90%.")

AA CPD genes at >=90%: 46
EA CPD genes at >=90%: 42

Overlap at >=90% stability: 0
[]

Still no overlap at 90%.


In [9]:
aa_cpd_80 = set(aa_cpd_all[aa_cpd_all["stability_fraction"] >= 0.5]["gene"].tolist())
ea_cpd_80 = set(ea_cpd_all[ea_cpd_all["stability_fraction"] >= 0.5]["gene"].tolist())

print("AA CPD genes at >=80%:", len(aa_cpd_80))
print("EA CPD genes at >=80%:", len(ea_cpd_80))

overlap_80 = aa_cpd_80 & ea_cpd_80
print(f"\nOverlap at >=80% stability: {len(overlap_80)}")
print(sorted(overlap_80))

from scipy.stats import hypergeom
M = 11663
n = len(aa_cpd_80)
N = len(ea_cpd_80)
k = len(overlap_80)
if k > 0:
    p = hypergeom.sf(k-1, M, n, N)
    print(f"\nP(>= {k} overlap by chance): {p:.6f}")
else:
    print("\nStill no overlap at 80%.")

AA CPD genes at >=80%: 61
EA CPD genes at >=80%: 406

Overlap at >=80% stability: 2
['MYT1', 'RUFY4']

P(>= 2 overlap by chance): 0.632173


In [10]:
print("AA CPD (smokers only) stats:")
print(pheno_cpd.describe())  # if not in this kernel, may need reload
print("\nEA CPD (smokers only) stats:")
print(pheno_cpd_ea.describe())

AA CPD (smokers only) stats:


NameError: name 'pheno_cpd' is not defined

In [11]:
aa_confounders = np.load(r"C:\Users\user\Downloads\GSE148375_clean\confounders_X_cpd.npy")
print("AA confounders shape:", aa_confounders.shape)
print("\nAA confounder column stats:")
for col_idx in range(12):
    col = aa_confounders[:, col_idx]
    print(f"Col {col_idx}: mean={col.mean():.3f}, std={col.std():.3f}, min={col.min():.3f}, max={col.max():.3f}")

AA confounders shape: (1459, 12)

AA confounder column stats:
Col 0: mean=0.000, std=28.866, min=-35.572, max=154.936
Col 1: mean=0.000, std=10.986, min=-34.065, max=41.587
Col 2: mean=-0.000, std=10.652, min=-22.403, max=40.642
Col 3: mean=-0.000, std=9.973, min=-27.830, max=38.016
Col 4: mean=0.000, std=9.435, min=-39.724, max=33.822
Col 5: mean=0.000, std=9.406, min=-29.871, max=45.845
Col 6: mean=-0.000, std=9.191, min=-34.918, max=35.780
Col 7: mean=0.000, std=8.952, min=-34.507, max=32.988
Col 8: mean=-0.000, std=8.804, min=-44.015, max=32.964
Col 9: mean=-0.000, std=8.682, min=-42.384, max=36.992
Col 10: mean=-0.000, std=1.000, min=-2.061, max=3.512
Col 11: mean=0.524, std=0.499, min=0.000, max=1.000


In [12]:
import pandas as pd

aa_dir = r"C:\Users\user\Downloads\GSE148375_clean"
ea_dir = r"C:\Users\user\Downloads\GSE148812_clean"

meta_aa = pd.read_csv(os.path.join(aa_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))
meta_aa["sample_id"] = meta_aa["sample_id"].astype(str)
meta_aa_smokers = meta_aa[meta_aa["smoking_status"]=="Smoker"]

meta_ea = pd.read_csv(os.path.join(ea_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))
meta_ea["sample_id"] = meta_ea["sample_id"].astype(str)
meta_ea_smokers = meta_ea[(meta_ea["smoking_status"]=="Smoker") & (meta_ea["cpd"].notna())]

print("AA CPD distribution (n=%d):" % len(meta_aa_smokers))
print(meta_aa_smokers["cpd"].describe())
print("\nEA CPD distribution (n=%d):" % len(meta_ea_smokers))
print(meta_ea_smokers["cpd"].describe())

AA CPD distribution (n=1459):
count    1459.000000
mean       26.571624
std         6.623071
min         1.000000
25%        20.000000
50%        30.000000
75%        30.000000
max        60.000000
Name: cpd, dtype: float64

EA CPD distribution (n=793):
count    793.000000
mean      27.832282
std        7.808042
min        7.000000
25%       20.000000
50%       30.000000
75%       30.000000
max       65.000000
Name: cpd, dtype: float64
